<a href="https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**1. Method Choice and Why**

I selected **Random Forest** Classification for this project because it can learn relationships between multiple features better than a fixed rule. My Week-4 baseline used simple thresholds based on impressions, CTR, and content freshness. Random Forest combines these signals automatically and can identify more complex patterns while still producing interpretable feature importance.

The goal is to predict whether a page should be prioritized for content refresh. The model will be compared directly against my Week-4 baseline using the same dataset and evaluation metric.

In [6]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Working directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Dataset loaded successfully!
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
import os
import pandas as pd

# Locate repository root
while not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if os.getcwd() == "/":
        raise FileNotFoundError("Dataset not found. Open this notebook from your GitHub repository.")
    os.chdir("..")

print("Working Directory:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Target variable
df["target"] = (df["trend_direction"] == "down").astype(int)

print(df.shape)
df.head()

Working Directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
(30000, 45)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,target
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


## 2. Split design

I used an 80/20 train-test split to evaluate the model. Eighty percent of the data is used for training, while twenty percent is reserved for testing. This provides an honest estimate of model performance on unseen data.

The baseline rule and the Random Forest model are evaluated using the same test set and the same evaluation metrics so that the comparison is fair.

In [8]:
from sklearn.model_selection import train_test_split

features = [
    "impressions_90d",
    "ctr",
    "days_since_last_update",
    "content_age_days",
    "word_count",
    "avg_position"
]

X = df[features].fillna(0)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 24000
Testing rows: 6000


## 3. Train + compare vs my baseline

The Random Forest model is trained using observable features only. The same test set is used to evaluate both the Week-4 baseline rule and the machine learning model.

The comparison uses the same evaluation metric and dataset to ensure fairness. The objective is to determine whether the machine learning model improves upon the manually designed baseline.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Train model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

# Week-4 baseline
baseline_score = (
    (X_test["impressions_90d"] >= 500).astype(int) +
    (X_test["ctr"] < 0.05).astype(int) +
    (X_test["days_since_last_update"] >= 180).astype(int)
)

baseline_pred = (baseline_score >= 2).astype(int)

comparison = pd.DataFrame({
    "Method": ["Baseline Rule", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred),
        precision_score(y_test, pred)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred),
        recall_score(y_test, pred)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred),
        f1_score(y_test, pred)
    ]
})

comparison

,Method,Accuracy,Precision,Recall,F1 Score
0,Baseline Rule,0.494167,0.669086,0.141065,0.233005
1,Random Forest,0.686000,0.698281,0.745716,0.721219


## 4. Errors and interpretation
The Random Forest model generally performs better than the manually designed baseline because it combines multiple features instead of relying on fixed thresholds. However, prediction errors still occur because search performance is influenced by seasonality, competition, user behavior, and other external factors that are not included in the dataset.

Feature importance indicates that impressions, CTR, and content freshness are among the strongest predictors. This model should be used as decision-support rather than as a replacement for human judgment.

In [10]:
# Feature importance
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("Feature Importance")
display(importance)

print("\nComparison Table")
display(comparison)

Feature Importance


,Feature,Importance
0,impressions_90d,0.273122
5,avg_position,0.248052
3,content_age_days,0.161535
4,word_count,0.161051
1,ctr,0.114526
2,days_since_last_update,0.041715



Comparison Table


,Method,Accuracy,Precision,Recall,F1 Score
0,Baseline Rule,0.494167,0.669086,0.141065,0.233005
1,Random Forest,0.686000,0.698281,0.745716,0.721219


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.